# x_true uncertainty in fit_trans: Domingo exon skipping
Cis gene **GFI1B**, GPU **cuda:7**. This mirrors the production modality stage (`publication_runs/domingo/load_modalities.py`): it rebuilds from `<label>_modality_exon_skip.yaml`, loads the cis fit, attaches the precomputed exon_skip subset and loads the production exon_skip NTC fit. Only `fit_trans` is rerun, once per x-input condition.

In [ ]:
%load_ext autotime
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.path.abspath('.'))   # run from testing/xtrue_uncertainty/
import xunc_helpers as H
from config_utils import normalize_stage_args, is_enabled
print(os.environ.get('CONDA_DEFAULT_ENV'), torch.cuda.is_available())

In [ ]:
# Parameters
CIS_GENE     = 'GFI1B'
LABEL_PREFIX = 'domingo_20260806'
STYPE        = 'exon_skip'
DEVICE           = 7   # GPU index on the node
TEST_OUTPUT_DIR  = '/cfs/klemming/projects/snic/lappalainen_lab1/users/Leah/CRISPRmodelling/BayesianModelling/xtrue_uncertainty'
X_DRAWS_PER_STEP = 1
REFIT            = False   # True: refit even if a test fit is already saved

CONDITIONS = H.CONDITIONS   # cell_median (old behaviour), cell_sample, guide_sample
device = f'cuda:{DEVICE}' if torch.cuda.is_available() else 'cpu'

## Rebuild the production modality model

In [ ]:
import yaml
sys.path.insert(0, str(H.REPO / 'publication_runs' / 'domingo'))
from load_modalities import attach_modality_precomputed

PROD_LABEL = f'{LABEL_PREFIX}_{CIS_GENE}'
LABEL = f'xunc_{PROD_LABEL}_{STYPE}'
model, cfg = H.build_modality_model(H.production_config('domingo', PROD_LABEL, f'modality_{STYPE}'), device)

with open(H.REPO / 'publication_runs' / 'domingo' / 'config_modalities.yaml') as f:
    spec = {m['stype']: m for m in yaml.safe_load(f)['modalities']}[STYPE]
precomputed_dir = os.path.join(cfg['model']['output_dir'], f'{PROD_LABEL}_modality_subset', STYPE)
MODALITY = attach_modality_precomputed(model, spec, precomputed_dir)

# production modality NTC fit (written by load_modalities.py into the gene's label dir)
model.load_ntc_fit(H.production_dir(cfg), modalities=[MODALITY])

mod = model.get_modality(MODALITY)
if 'feature_id' in mod.feature_meta.columns:   # plot/look up by junction-triplet id
    mod.feature_names = list(mod.feature_meta['feature_id'].values)
print(mod)

## How noisy is x?
The per-cell posterior SD of log2 `x_true`, compared with the within-guide spread of the per-cell medians (the ordering the old fit treated as exact) and the spread between guides.

In [ ]:
x_unc = H.x_uncertainty_report(model)

## Fit trans under each x-input condition
Each condition is a deep copy of the prepared model, refit with `fit_trans(x_level=..., x_uncertainty=...)` and saved to `TEST_OUTPUT_DIR/<label>_<condition>/`. If that directory already has a saved fit, it is loaded instead (set `REFIT = True` to refit). Production outputs are never written.

In [ ]:
fit_kwargs = dict(tolerance=0, function_type=spec['function_type'],
                  min_denominator=spec.get('min_denominator', 0),
                  x_draws_per_step=X_DRAWS_PER_STEP)
print(fit_kwargs)

models, summaries = H.fit_conditions(model, fit_kwargs, TEST_OUTPUT_DIR, LABEL,
                                     conditions=CONDITIONS, modality_name=MODALITY, refit=REFIT)

## Compare conditions
Counts of dependent genes, by active Hill component, then each gene's Hill coefficients against `cell_median` (the old behaviour).

In [ ]:
H.dependence_table(summaries)

In [ ]:
fig = H.plot_n_shift(summaries)
plt.show()

## Trans fits, all genes
One panel per feature, with every condition's fitted curve and 95% CI overlaid. Panel titles give `which_active` per condition (cm = cell_median, cs = cell_sample, gs = guide_sample). All pages also go to `overlays.pdf`.

In [ ]:
features = sorted(summaries['cell_median']['feature'])
H.plot_condition_overlays(models, summaries, features, modality_name=MODALITY,
                          pdf_path=os.path.join(TEST_OUTPUT_DIR, f'{LABEL}_overlays.pdf'),
                          use_log2fc=False)

## Data + fit per condition (saved, not shown)
`plot_xy_data` for every gene × condition → `TEST_OUTPUT_DIR/<label>_xy/<condition>/<gene>.png`. In every condition the x positions are the per-cell posterior-median `x_true`, including `guide_sample`, whose fit actually used guide-level x. Pass `features=[...]` to `plot_xy_data` directly to look at a few inline.

In [ ]:
H.save_xy_plots(models, features, os.path.join(TEST_OUTPUT_DIR, f'{LABEL}_xy'),
                modality_name=MODALITY, window=50, min_counts=0, show_correction='corrected', legend_outside=True, figsize=(5, 3))